<a href="https://colab.research.google.com/github/yvn2000/NLP_CW1/blob/main/material/F21NL_CW1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data download

In [7]:
!wget -O wikitext-filtered-full.zip "https://www.dropbox.com/scl/fi/ibd4cmixckghx6hhb361c/wikitext-filtered-full.zip?rlkey=q71cebf0k5fvvwhmcntoswzhq&dl=1"
!wget -O wikitext-filtered-10k.zip "https://www.dropbox.com/scl/fi/ek174r3sg7qjx0aa9atop/wikitext-filtered-10k.zip?rlkey=zy6jqxv6qsc16lr9qm3ki9uhf&dl=1"

--2026-10-11 14:08:17--  https://www.dropbox.com/scl/fi/ibd4cmixckghx6hhb361c/wikitext-filtered-full.zip?rlkey=q71cebf0k5fvvwhmcntoswzhq&dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.4.18, 2620:100:6019:18::a27d:412
Connecting to www.dropbox.com (www.dropbox.com)|162.125.4.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://uc2d8bfdd5db537831c31cabfa46.dl-eu.dropboxusercontent.com/cd/0/inline/DJxx9EFsGlEytCA_O5cHaQi5WjgqNO0tTgOqKoWUb4N2vAhVi-y8gzjpapmmDlIPphqzTBvkKSpdxzR56h-uLTFKFtMQlzfy5XP3xkrW1S4MXbHJSFK9-Eu1m5QPr_hIHh_b5q-ql-JL90dIVxA1L7z4/file?dl=1# [following]
--2026-10-11 14:08:18--  https://uc2d8bfdd5db537831c31cabfa46.dl-eu.dropboxusercontent.com/cd/0/inline/DJxx9EFsGlEytCA_O5cHaQi5WjgqNO0tTgOqKoWUb4N2vAhVi-y8gzjpapmmDlIPphqzTBvkKSpdxzR56h-uLTFKFtMQlzfy5XP3xkrW1S4MXbHJSFK9-Eu1m5QPr_hIHh_b5q-ql-JL90dIVxA1L7z4/file?dl=1
Resolving uc2d8bfdd5db537831c31cabfa46.dl-eu.dropboxusercontent.com (uc2d8bfdd5db537831c31cabfa46.dl-eu.dropbo

In [ ]:
!unzip wikitext-filtered-full.zip
!unzip wikitext-filtered-10k.zip

In [ ]:
!pip install datasets
import datasets

In [ ]:
from datasets import load_dataset, Dataset

def load_dataset():
  wikitext_small = "wikitext-filtered-10k"
  wikitext_large = "wikitext-filtered-full"

  dataset_small = Dataset.load_from_disk(wikitext_small)
  dataset_large = Dataset.load_from_disk(wikitext_large)
  print("wikitext_small: {} docs, wikitext_large: {} docs".format(len(dataset_small), len(dataset_large)))
  return dataset_small, dataset_large

wikitext_small, wikitext_large = load_dataset()

# Callback: EpochLossLogger

In [ ]:
!pip install -q gensim

In [ ]:
from gensim.models.callbacks import CallbackAny2Vec

class EpochLossLogger(CallbackAny2Vec):
    """Callback to log loss after each epoch."""

    def __init__(self):
        self.epoch = 0
        self.prev_loss = 0.0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        epoch_loss = loss - self.prev_loss

        print(
            f"Epoch {self.epoch + 1:02d} | "
            f"epoch loss = {epoch_loss:.2f} | "
            f"cumulative loss = {loss:.2f}"
        )

        self.prev_loss = loss
        self.epoch += 1

loss_logger = EpochLossLogger()

The callback is used as follows:


```
model = Word2Vec(
    ...
    compute_loss = True,
    callbacks=[loss_logger],
    ...
    )
```



In [ ]:
"""
Two Goals:
1. Semantic Relatedness of two words (e.g., how close is the word “doctor” to “nurse”?).
2. Word Analogy (e.g., “apple is to tree as grape is to _ ?”, where the missing word should be “vine”.
"""